# Chapitre 5 — Analyser : cinq questions métier

**Ticket : SM-006**

## Objectif

Répondre aux cinq questions métier de SenMarket :

1. Où réalisons-nous notre chiffre d'affaires ?
2. Quels produits se vendent le mieux ?
3. Combien dépense un client en moyenne par commande ?
4. Quelle proportion de visiteurs commence un paiement ?
5. Les clients reviennent-ils après leur première commande ?

## Définitions métier

- **Chiffre d'affaires (CA)** : somme de `montant_xof` des lignes de commande nettoyées, sans filtrer selon le statut de la commande ou du paiement.
- **Panier moyen** : moyenne des montants totaux par commande.
- **Taux de conversion** : proportion des sessions ayant au moins un événement `debut_paiement`.
- **Ville inconnue** : les valeurs manquantes sont regroupées sous `Inconnue`, séparément des villes réelles.
- **Cohorte** : groupe de clients ayant réalisé leur première commande durant le même mois.
- **Rétention** : proportion des clients d'une cohorte qui commandent de nouveau au cours des mois suivants.

Les sessions sont comptées une seule fois par étape du tunnel.

In [2]:
from pathlib import Path
import pandas as pd
from IPython.display import display

from senmarket.analyse import (
    charger_donnees,
    analyser_ca_geographique,
    analyser_top_produits,
    analyser_panier_moyen,
    analyser_conversion,
    analyser_cohortes,
    analyser_paiements,
)

DOSSIER = Path("../data/processed/date=2026-11-02")

donnees = charger_donnees(DOSSIER)

for nom, df in donnees.items():
    print(f"{nom}: {len(df):,} lignes")

clients: 66,666 lignes
commandes: 80,000 lignes
lignes_commande: 200,000 lignes
produits: 40 lignes
navigation: 399,984 lignes
paiements: 78,400 lignes


## 1. Où réalisons-nous notre chiffre d'affaires ?

Nous calculons le CA à partir des lignes de commande, puis nous les relions aux commandes et aux clients.

L'analyse est effectuée par ville et par région.

**Contrôle :** la somme du CA de toutes les villes, y compris `Inconnue`, doit être égale au CA total.

In [3]:
resultat_geo = analyser_ca_geographique(
    donnees["lignes_commande"],
    donnees["commandes"],
    donnees["clients"],
)

print(f"CA total : {resultat_geo['ca_total']:,.0f} XOF")

print("\nCA par ville :")
display(resultat_geo["ca_ville"])

print("\nCA par région :")
display(resultat_geo["ca_region"])

assert abs(
    resultat_geo["ca_ville"]["ca_xof"].sum()
    - resultat_geo["ca_total"]
) < 0.01

print("Invariant vérifié : le CA par ville correspond au CA total.")

CA total : 108,417,563,344 XOF

CA par ville :


,ville,ca_xof
0,Rufisque,9985490350
1,Dakar,9203236906
2,Pikine,8610870117
3,Guediawaye,8208656780
4,Ziguinchor,4431621249
5,Salemata,3075479185
6,Fatick,3034751706
7,Foundiougne,2879742016
8,Oussouye,2790114489
9,Medina Yoro Foulah,2196292253



CA par région :


,region,ca_xof
0,Dakar,36008254153
1,Ziguinchor,9189726970
2,Kedougou,6121633291
3,Fatick,5914493722
4,Kaolack,5549560003
5,Sédhiou,5404238264
6,Louga,5303904765
7,Thies,5155616525
8,Kaffrine,5153606582
9,Diourbel,5113758914


Invariant vérifié : le CA par ville correspond au CA total.


## 2. Quels produits se vendent le mieux ?

Deux classements sont construits :

- le classement par quantité vendue ;
- le classement par chiffre d'affaires.

Ces classements peuvent être différents : un produit peu cher peut être vendu en grande quantité, tandis qu'un produit plus cher peut générer davantage de CA avec moins de ventes.

In [4]:
resultat_produits = analyser_top_produits(
    donnees["lignes_commande"],
    donnees["produits"],
)

print("Top 10 par quantité vendue :")
display(resultat_produits["top_quantite"])

print("Top 10 par chiffre d'affaires :")
display(resultat_produits["top_ca"])

Top 10 par quantité vendue :


,produit_id,nom_produit,quantite_vendue
0,PRD0000001,Produit 1,81709
1,PRD0000002,Produit 2,40443
2,PRD0000003,Produit 3,26804
3,PRD0000004,Produit 4,20736
4,PRD0000005,Produit 5,16300
5,PRD0000006,Produit 6,13824
6,PRD0000007,Produit 7,11416
7,PRD0000008,Produit 8,10296
8,PRD0000009,Produit 9,9098
9,PRD0000010,Produit 10,8099


Top 10 par chiffre d'affaires :


,produit_id,nom_produit,ca_xof
0,PRD0000001,Produit 1,38577876726
1,PRD0000004,Produit 4,9282670298
2,PRD0000002,Produit 2,4863344435
3,PRD0000007,Produit 7,4264158575
4,PRD0000009,Produit 9,3955521347
5,PRD0000003,Produit 3,3796932739
6,PRD0000005,Produit 5,3423912509
7,PRD0000012,Produit 12,3240461206
8,PRD0000010,Produit 10,2934562191
9,PRD0000008,Produit 8,2932124452


## 3. Combien dépense un client en moyenne par commande ?

Le panier moyen est calculé en deux étapes :

1. Additionner les montants des lignes de chaque commande.
2. Calculer la moyenne de ces totaux.

Nous comparons ce résultat à la moyenne des montants par ligne de commande.

Les deux valeurs sont différentes car une commande peut contenir plusieurs lignes. La moyenne par ligne ne représente donc pas la dépense moyenne par commande.

In [5]:
resultat_panier = analyser_panier_moyen(
    donnees["lignes_commande"]
)

print(
    f"Panier moyen : "
    f"{resultat_panier['panier_moyen']:,.0f} XOF"
)

print(
    f"Moyenne par ligne : "
    f"{resultat_panier['moyenne_par_ligne']:,.0f} XOF"
)

display(resultat_panier["totaux_commandes"].head())

Panier moyen : 1,477,079 XOF
Moyenne par ligne : 542,088 XOF


,commande_id,total_commande_xof
0,CMD0000001,974378
1,CMD0000002,2658652
2,CMD0000003,398537
3,CMD0000004,2127723
4,CMD0000005,2072625


## 4. Combien de visiteurs commencent un paiement ?

Nous utilisons la table `navigation`.

Chaque session est comptée une seule fois, même si elle contient plusieurs événements de la même catégorie.

Le taux de conversion est calculé ainsi :

**Taux de conversion = sessions avec un début de paiement / nombre total de sessions × 100**

Nous observons également les trois étapes du tunnel : vue, panier et début de paiement.

In [6]:
resultat_conversion = analyser_conversion(
    donnees["navigation"]
)

print("Nombre total de sessions :", resultat_conversion["total_sessions"])
print("Sessions avec une vue :", resultat_conversion["sessions_vue"])
print("Sessions avec un panier :", resultat_conversion["sessions_panier"])
print(
    "Sessions avec un début de paiement :",
    resultat_conversion["sessions_paiement"],
)

print(
    f"\nTaux de conversion : "
    f"{resultat_conversion['taux_conversion']:.2f} %"
)

display(resultat_conversion["tunnel"])

Nombre total de sessions : 133334
Sessions avec une vue : 133334
Sessions avec un panier : 51458
Sessions avec un début de paiement : 11913

Taux de conversion : 8.93 %


,etape,sessions
0,Vue,133334
1,Panier,51458
2,Début paiement,11913


## 5. Les clients reviennent-ils ?

Nous classons les clients selon le mois de leur première commande.

La matrice de rétention indique le pourcentage de clients de chaque cohorte qui réalisent au moins une commande au cours des mois suivants.

- Mois 0 : première commande, donc 100 %.
- Mois 1 : clients ayant commandé de nouveau le mois suivant.
- Mois 2 : clients ayant commandé deux mois après leur première commande.

La somme des tailles des cohortes doit être égale au nombre de clients ayant réalisé au moins une commande.

In [7]:
resultat_cohortes = analyser_cohortes(
    donnees["commandes"]
)

print("Nombre de clients ayant commandé :", resultat_cohortes["nombre_clients"])

print("\nTaille des cohortes :")
display(resultat_cohortes["taille_cohortes"])

print("\nMatrice de rétention (%) :")
display(resultat_cohortes["retention"].round(1))

assert (
    resultat_cohortes["taille_cohortes"]["nombre_clients"].sum()
    == resultat_cohortes["nombre_clients"]
)

Nombre de clients ayant commandé : 30139

Taille des cohortes :


,cohorte,nombre_clients
0,2025-11,314
1,2025-12,1024
2,2026-01,1500
3,2026-02,1756
4,2026-03,2174
5,2026-04,2499
6,2026-05,2927
7,2026-06,2971
8,2026-07,3437
9,2026-08,3635



Matrice de rétention (%) :


mois_depuis_cohorte,0,1,2,3,4,5,6,7,8,9,10,11,12
cohorte,,,,,,,,,,,,,
2025-11,100.0,19.1,23.6,20.4,24.2,20.4,22.0,25.2,21.7,22.0,22.0,21.0,3.8
2025-12,100.0,15.6,15.2,18.3,16.4,17.5,15.5,16.8,17.3,16.5,16.8,1.7,0.0
2026-01,100.0,14.1,14.7,14.0,13.6,14.3,14.2,14.2,14.2,15.6,1.5,0.0,0.0
2026-02,100.0,12.2,11.8,13.2,12.5,13.9,12.3,12.5,13.6,0.7,0.0,0.0,0.0
2026-03,100.0,12.5,13.5,11.7,12.6,11.4,10.4,12.0,0.9,0.0,0.0,0.0,0.0
2026-04,100.0,11.0,11.1,11.7,11.4,10.0,11.3,0.8,0.0,0.0,0.0,0.0,0.0
2026-05,100.0,9.4,10.2,11.0,10.2,10.5,0.9,0.0,0.0,0.0,0.0,0.0,0.0
2026-06,100.0,10.1,10.0,9.1,10.0,0.8,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2026-07,100.0,9.5,9.3,9.7,0.7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Bonus — CA commandé et CA encaissé

Nous comparons :

- le CA commandé : somme des montants de toutes les lignes de commande ;
- le CA encaissé : somme des paiements dont le statut indique une réussite.

Nous calculons également le nombre de commandes sans paiement réussi.

Cette différence permet d'identifier les écarts entre les ventes enregistrées et les paiements effectivement réussis.

In [8]:
resultat_paiements = analyser_paiements(
    donnees["lignes_commande"],
    donnees["paiements"],
)

print(f"CA commandé : {resultat_paiements['ca_commande']:,.0f} XOF")
print(f"CA encaissé : {resultat_paiements['ca_encaisse']:,.0f} XOF")
print(f"Écart : {resultat_paiements['ecart_ca']:,.0f} XOF")

print(
    "Commandes sans paiement réussi :",
    resultat_paiements["commandes_sans_paiement_reussi"],
)

CA commandé : 108,417,563,344 XOF
CA encaissé : 95,688,897,278 XOF
Écart : 12,728,666,066 XOF
Commandes sans paiement réussi : 2748


## Synthèse des résultats

À compléter après exécution des analyses.

### 1. Chiffre d'affaires géographique
- Ville qui génère le plus de CA :
- Région qui génère le plus de CA :
- Part du CA associée à `Inconnue` :

### 2. Produits
- Produit le plus vendu en quantité :
- Produit qui génère le plus de CA :
- Pourquoi les classements sont-ils différents ?

### 3. Panier moyen
- Panier moyen par commande :
- Moyenne par ligne :
- Pourquoi les deux valeurs diffèrent-elles ?

### 4. Conversion
- Nombre total de sessions :
- Nombre de sessions avec début de paiement :
- Taux de conversion :

### 5. Fidélisation
- Nombre de clients ayant commandé :
- Que révèle la rétention au mois 1 ?

### Bonus
- CA commandé :
- CA encaissé :
- Nombre de commandes sans paiement réussi :

### Réflexion

**Pourquoi les Top 10 par quantité et par CA peuvent-ils différer ?**

Parce que le prix des produits varie. Un produit peu coûteux peut être acheté en grande quantité sans générer le CA le plus élevé.

**Que nous apprend la rétention au mois 1 sur la qualité des données générées ?**

Elle permet d'observer si les clients reviennent commander. Une rétention très faible ou très élevée peut aussi révéler des comportements de génération artificiels. Il faut donc interpréter ce résultat comme une caractéristique des données simulées, et non comme une mesure réelle de fidélité client.